In [1]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

BEST_MODEL = (
    PROJECT_ROOT
    / "results"
    / "baseline_yolo26n"
    / "weights"
    / "best.pt"
)

print("Model path:", BEST_MODEL)
print("Model exists:", BEST_MODEL.exists())

Model path: /Users/priyansh/Downloads/airport-baggage-screening/results/baseline_yolo26n/weights/best.pt
Model exists: True


In [2]:
model = YOLO(str(BEST_MODEL))

print("Baseline model loaded successfully.")

Baseline model loaded successfully.


In [3]:
DATA_YAML = PROJECT_ROOT / "data" / "project_data.yaml"

metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=8,
    device="mps",
    workers=0,
    plots=True
)

Ultralytics 8.4.118 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
YOLO26n summary (fused): 122 layers, 2,377,176 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 309.5±124.2 MB/s, size: 107.4 KB)
val: Scanning /Users/priyansh/Downloads/airport-baggage-screening/data/labels/val.cache... 1373 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1373/1373 274.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 172/172 7.8it/s 21.9s0.1ss
                   all       1373       1794      0.867      0.734      0.796      0.629
                 Baton        143        148      0.902      0.892      0.933      0.725
                Pliers        142        168      0.718      0.621      0.688      0.455
                Hammer        135        142      0.953      0.866      0.919      0.822
             Powerbank        147        154      0.912      0.799      0.848      0.664
              S

In [4]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

Precision: 0.8669765690907907
Recall: 0.7337984234795751
mAP@50: 0.7964639849857705
mAP@50-95: 0.6286039015908174


In [5]:
print("Class names:")
print(model.names)

Class names:
{0: 'Baton', 1: 'Pliers', 2: 'Hammer', 3: 'Powerbank', 4: 'Scissors', 5: 'Wrench', 6: 'Gun', 7: 'Bullet', 8: 'Sprayer', 9: 'HandCuffs', 10: 'Knife', 11: 'Lighter'}


In [6]:
for class_id, class_name in model.names.items():

    precision = metrics.box.p[class_id]
    recall = metrics.box.r[class_id]
    ap50 = metrics.box.ap50[class_id]
    ap = metrics.box.ap[class_id]

    print(
        f"{class_id:2d} | "
        f"{class_name:12s} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"mAP50: {ap50:.3f} | "
        f"mAP50-95: {ap:.3f}"
    )

 0 | Baton        | Precision: 0.902 | Recall: 0.892 | mAP50: 0.933 | mAP50-95: 0.725
 1 | Pliers       | Precision: 0.718 | Recall: 0.621 | mAP50: 0.688 | mAP50-95: 0.455
 2 | Hammer       | Precision: 0.953 | Recall: 0.866 | mAP50: 0.919 | mAP50-95: 0.822
 3 | Powerbank    | Precision: 0.912 | Recall: 0.799 | mAP50: 0.848 | mAP50-95: 0.664
 4 | Scissors     | Precision: 0.855 | Recall: 0.645 | mAP50: 0.726 | mAP50-95: 0.499
 5 | Wrench       | Precision: 0.856 | Recall: 0.726 | mAP50: 0.805 | mAP50-95: 0.659
 6 | Gun          | Precision: 0.876 | Recall: 0.748 | mAP50: 0.776 | mAP50-95: 0.502
 7 | Bullet       | Precision: 0.867 | Recall: 0.684 | mAP50: 0.786 | mAP50-95: 0.627
 8 | Sprayer      | Precision: 0.913 | Recall: 0.801 | mAP50: 0.868 | mAP50-95: 0.745
 9 | HandCuffs    | Precision: 0.941 | Recall: 0.936 | mAP50: 0.964 | mAP50-95: 0.872
10 | Knife        | Precision: 0.764 | Recall: 0.539 | mAP50: 0.613 | mAP50-95: 0.468
11 | Lighter      | Precision: 0.848 | Recall: 0.549 |

In [7]:
TEST_DIR = PROJECT_ROOT / "data" / "images" / "test"

test_results = model.predict(
    source=str(TEST_DIR),
    imgsz=640,
    conf=0.25,
    device="mps",
    save=True,
    project=str(PROJECT_ROOT / "results"),
    name="baseline_test_predictions",
    exist_ok=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1373 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg: 512x640 1 Baton, 16.2ms
image 2/1373 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00035.jpg: 512x640 1 Baton, 189.8ms
image 3/1373 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00039.jpg: 640x640 1 Baton, 28.5ms
image 4/1373 /Users/priyansh/Downloads/airport-baggage-screening/data/images/te

In [8]:
test_images = sorted(TEST_DIR.glob("*.jpg"))

sample = test_images[0]

print(sample)

/Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg


In [9]:
result = model.predict(
    source=str(sample),
    imgsz=640,
    conf=0.25,
    device="mps"
)

result[0].show()


image 1/1 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg: 512x640 1 Baton, 47.6ms
Speed: 7.1ms preprocess, 47.6ms inference, 4.0ms postprocess per image at shape (1, 3, 512, 640)


In [10]:
result = model.predict(
    source=str(sample),
    imgsz=640,
    conf=0.50,
    device="mps"
)

result[0].show()


image 1/1 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg: 512x640 1 Baton, 39.2ms
Speed: 13.2ms preprocess, 39.2ms inference, 5.0ms postprocess per image at shape (1, 3, 512, 640)


In [11]:
result = model.predict(
    source=str(sample),
    imgsz=640,
    conf=0.75,
    device="mps"
)

result[0].show()


image 1/1 /Users/priyansh/Downloads/airport-baggage-screening/data/images/test/PID_xray_00024.jpg: 512x640 1 Baton, 44.1ms
Speed: 39.6ms preprocess, 44.1ms inference, 4.6ms postprocess per image at shape (1, 3, 512, 640)
